# Natural Language Processing with Disaster Tweets

In [2]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
train_df.shape 

(7613, 5)

In [4]:
print("Brakujące wartości:")
print(train_df.isnull().sum())

print("\nBalans klas (%):")
print(train_df['target'].value_counts(normalize=True).round(3) * 100)

keyword_stats = train_df.groupby('keyword')['target'].agg(['count', 'mean'])
keyword_stats = keyword_stats[keyword_stats['count'] >= 20] 
print("\nTop 10 słów (najwyższy odsetek prawdziwych katastrof):")
print(keyword_stats.sort_values('mean', ascending=False).head(10))

Brakujące wartości:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Balans klas (%):
target
0    57.0
1    43.0
Name: proportion, dtype: float64

Top 10 słów (najwyższy odsetek prawdziwych katastrof):
                   count      mean
keyword                           
debris                37  1.000000
derailment            39  1.000000
wreckage              39  1.000000
outbreak              40  0.975000
oil%20spill           38  0.973684
typhoon               38  0.973684
suicide%20bombing     33  0.969697
suicide%20bomber      31  0.967742
bombing               29  0.931034
suicide%20bomb        35  0.914286


## czyszczenie danych

In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Czyszczenie kolumny text
train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

# Czyszczenie kolumny keyword
train_df['keyword'] = train_df['keyword'].fillna('').str.replace('%20', ' ')
test_df['keyword'] = test_df['keyword'].fillna('').str.replace('%20', ' ')

# Stworzenie połączonej cechy: keyword + wyczyszczony tekst
train_df['final_text'] = train_df['keyword'] + " " + train_df['clean_text']
test_df['final_text'] = test_df['keyword'] + " " + test_df['clean_text']

print(train_df[['text', 'final_text']].head())

                                                text  \
0  Our Deeds are the Reason of this #earthquake M...   
1             Forest fire near La Ronge Sask. Canada   
2  All residents asked to 'shelter in place' are ...   
3  13,000 people receive #wildfires evacuation or...   
4  Just got sent this photo from Ruby #Alaska as ...   

                                          final_text  
0   our deeds are the reason of this earthquake m...  
1              forest fire near la ronge sask canada  
2   all residents asked to shelter in place are b...  
3   13000 people receive wildfires evacuation ord...  
4   just got sent this photo from ruby alaska as ...  


In [6]:
train_df.head()

,id,keyword,location,text,target,clean_text,final_text
0,1,,NaN,Our Deeds are the Reason of this #earthquake M...,1,our deeds are the reason of this earthquake ma...,our deeds are the reason of this earthquake m...
1,4,,NaN,Forest fire near La Ronge Sask. Canada,1,forest fire near la ronge sask canada,forest fire near la ronge sask canada
2,5,,NaN,All residents asked to 'shelter in place' are ...,1,all residents asked to shelter in place are be...,all residents asked to shelter in place are b...
3,6,,NaN,"13,000 people receive #wildfires evacuation or...",1,13000 people receive wildfires evacuation orde...,13000 people receive wildfires evacuation ord...
4,7,,NaN,Just got sent this photo from Ruby #Alaska as ...,1,just got sent this photo from ruby alaska as s...,just got sent this photo from ruby alaska as ...


## Baseline: regresja logistyczna TD-IDF

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [16]:
X = train_df['final_text']
y = train_df['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__max_features': [5000, 10000, None],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [3.0, 5.0, 6.0, 7.0, 2.0, 4.0]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)

print("Rozpoczynam strojenie hiperparametrów...")
grid.fit(X_train, y_train)

print(f"Najlepsze F1 Score na walidacji (CV): {grid.best_score_:.4f}")
print("Najlepsze parametry:")
for param, value in grid.best_params_.items():
    print(f"- {param}: {value}")

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_val)

print("\nOstateczny wynik na X_val:")
print("F1 Score:", round(f1_score(y_val, y_pred_best), 4))
print(classification_report(y_val, y_pred_best))

Rozpoczynam strojenie hiperparametrów...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Najlepsze F1 Score na walidacji (CV): 0.7506
Najlepsze parametry:
- clf__C: 3.0
- tfidf__max_features: 5000
- tfidf__ngram_range: (1, 1)

Ostateczny wynik na X_val:
F1 Score: 0.7773
              precision    recall  f1-score   support

           0       0.82      0.87      0.84       869
           1       0.81      0.74      0.78       654

    accuracy                           0.82      1523
   macro avg       0.82      0.81      0.81      1523
weighted avg       0.82      0.82      0.82      1523



## SVC

In [19]:
from sklearn.svm import SVC

pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', SVC(random_state=42))
])


param_grid_svm = {
    'tfidf__max_features': [5000, 10000, None],
    'tfidf__ngram_range': [(1, 1)],
    'svm__C': [1.0, 1.5],
    'svm__kernel': ['linear']
}

grid_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=5, scoring='f1', n_jobs=-1, verbose=1)

print("Rozpoczynam strojenie SVM...")
grid_svm.fit(X_train, y_train)

print(f"\nNajlepsze parametry SVM: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

y_pred_svm = best_svm.predict(X_val)
print("\nSVM F1 Score na X_val:", round(f1_score(y_val, y_pred_svm), 4))
print("\nRaport z Klasyfikacji (SVM):\n", classification_report(y_val, y_pred_svm))

Rozpoczynam strojenie SVM...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Najlepsze parametry SVM: {'svm__C': 1.0, 'svm__kernel': 'linear', 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}

SVM F1 Score na X_val: 0.7727

Raport z Klasyfikacji (SVM):
               precision    recall  f1-score   support

           0       0.81      0.88      0.85       869
           1       0.82      0.73      0.77       654

    accuracy                           0.82      1523
   macro avg       0.82      0.81      0.81      1523
weighted avg       0.82      0.82      0.81      1523



## transformer